## Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

## Pydantic
Pydantic models provide the richest feature set with field validation, description and nested structures.

In [9]:
import os 
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("groq")
model=init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000026082BBBA50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000026082BBB190>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [10]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str=Field(description="The title of the movie")
    year: int=Field(description="This year the movie was released")
    director: str=Field(description="The director name of the movie")
    rating:float=Field(description="The Movie rating out of 10")

In [11]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000026082BBBA50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000026082BBB190>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director name of the movie', 'type': 'string'}, 'rating': {'description': 'The Movie rating

In [12]:
response=model_with_structure.invoke("provide details of movie inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message output alongside parsed structure

In [13]:
from pydantic import BaseModel, Field
"... is an optional field"

class Movie(BaseModel):
    title: str=Field(...,description="The title of the movie")
    year: int=Field(...,description="This year the movie was released")
    director: str=Field(...,description="The director name of the movie")
    rating:float=Field(...,description="The Movie rating out of 10")

model_with_structure=model.with_structured_output(Movie, include_raw=True)
response=model_with_structure.invoke("provide details of movie inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me check the available functions. There\'s a Movie function that requires title, year, director, and rating. I need to recall the information about Inception. The title is "Inception", the director is Christopher Nolan, it was released in 2010, and the rating is 8.8 on IMDb. I should structure the JSON with these parameters. Let me make sure all required fields are included: title, year, director, rating. Yes, that\'s all there. I\'ll format the tool call accordingly.\n', 'tool_calls': [{'id': 'g12kk0j25', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 171, 'prompt_tokens': 224, 'total_tokens': 395, 'completion_time': 0.290170679, 'completion_tokens_details': {'reasoning_tokens': 123}, 'prompt_t

# Nested Structure

In [14]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")
model_with_structure=model.with_structured_output(MovieDetails)
response =model_with_structure.invoke("provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Balthazar')], genres=['Action', 'Science Fiction', 'Thriller'], budget=160.0)

# TypeDict

TypeDict provides a simple alternative using Python's built-in typing, ideal when you don't need runtime validation.

In [16]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details"""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_with_typedict=model.with_structured_output(MovieDict)
response=model_with_typedict.invoke("please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}

In [18]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: Annotated[float | None, "Budget in millions USD"]
model_with_structure=model.with_structured_output(MovieDetails)
response =model_with_structure.invoke("provide details about the movie Inception")
response

{'budget': 160,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'},
  {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'},
  {'name': 'Ellen Page', 'role': 'Ariadne'}],
 'genres': ['Science Fiction', 'Action', 'Thriller'],
 'title': 'Inception',
 'year': 2010}